<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/05_Intel_Image_Classification_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip -d ./intel_images

In [ ]:
import os

base_dir = './intel_images'
print(os.listdir(base_dir))

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor()
])

train_dir = './intel_images/seg_train/seg_train'
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

class_names = train_dataset.classes
print(class_names)
print(type(class_names), len(class_names))

print(np.array(class_names).shape)

In [ ]:
print(len(train_dataset), len(train_loader))
print(type(train_dataset), type(train_loader))
img, label = train_dataset[0]
print(img.shape)
image, label = next(iter(train_loader))
print(image.shape)

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(1, 4, figsize=(12, 4))

for i in range(4):
    img = images[i]
    label = labels[i].item()

    img_permuted = img.permute(1, 2, 0)

    axes[i].imshow(img_permuted)
    axes[i].set_title(class_names[label])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(128 * 18 * 18, 512)
        self.fc2 = nn.Linear(512, 6)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN()

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 3

for epoch in range(epochs):
    running_loss, epoch_loss = 0.0, 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        epoch_loss += loss.item()

        if (i+1) % 500 == 0:
            print(f"Batch {i+1} | Loss: {running_loss/500:.4f}")
            running_loss = 0.0

    print(f"==> Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:
import torchvision.models as models

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

in_features = resnet.fc.in_features
print(in_features)

resnet.fc = nn.Linear(in_features, 6)

resnet = resnet.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer_resnet = optim.Adam(resnet.parameters(), lr=0.0001)

In [ ]:
epochs = 3

resnet.train()

for epoch in range(epochs):
    running_loss, epoch_loss = 0.0, 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer_resnet.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_resnet.step()
        running_loss += loss.item()
        epoch_loss += loss.item()

        if (i+1) % 500 == 0:
            print(f"Batch {i+1} | Loss: {running_loss/500:.4f}")
            running_loss = 0.0

    print(f"==> ResNet Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:
test_dir = './intel_images/seg_test/seg_test'
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

resnet.eval()

correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = resnet(images)

        _, predicted = torch.max(outputs.data, 1)

        total += len(labels)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor()
])

train_dir = './intel_images/seg_train/seg_train'
test_dir = './intel_images/seg_test/seg_test'

augmented_train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform_augmented)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_dataloader_aug = DataLoader(augmented_train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(len(train_dataloader_aug), len(test_dataloader))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
model_aug = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_feature = model_aug.fc.in_features
model_aug.fc = nn.Linear(in_features, 6)
model_aug = model_aug.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_aug.parameters(), lr=0.0001)

In [ ]:
epochs = 3

model_aug.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for i, (images, labels) in enumerate(train_dataloader_aug):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_aug(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

print(f"==> Augmented ResNet | Epoch {epoch+1} | Loss: {epoch_loss/len(train_dataloader_aug):.4f}")

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

images, labels = next(iter(test_loader))

model_aug.eval()
with torch.no_grad():
    outputs = model_aug(images.to(device))
    _, preds = torch.max(outputs, 1)

images = images.cpu()
labels = labels.numpy()
preds = preds.cpu().numpy()

random_indices = np.random.choice(len(images), 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, idx in enumerate(random_indices):
    img = images[idx].permute(1, 2, 0)
    true_label = class_names[labels[idx]]
    pred_label = class_names[preds[idx]]

    color = 'blue' if true_label == pred_label else 'red'

    axes[i].imshow(img)
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=14)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

all_preds, all_labels = [], []

model_aug.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model_aug(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)

plt.xlabel('Predicted (Model)', fontsize=12)
plt.ylabel('True (Actual)', fontsize=12)
plt.title('Intel Image Classification - Confusion Matrix', fontsize=15)
plt.show()